# SIA — transformer fine-tune on Colab (GPU)

Runs the submission-grade backend: **DeBERTa-v3-small** fine-tuned on the self-supervised pseudo-labels.

**Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all* (~15–25 min).

In [ ]:
!git clone https://github.com/amrendra1423/support-integrity-auditor.git sia_repo
%cd sia_repo

In [ ]:
!pip -q install transformers==4.46.3 sentencepiece==0.2.0 peft==0.13.2

In [ ]:
# dataset ships in the repo (data/customer_support_tickets.csv)
!python train_pipeline.py --data data/customer_support_tickets.csv \
    --backend transformer --outdir artifacts_hf --skip-ablation

In [ ]:
import json
m = json.load(open('artifacts_hf/metrics.json'))
print(json.dumps({k: m[k] for k in ('accuracy','macro_f1','backend')}, indent=2))
print(json.dumps(m['per_class'], indent=2))
print('VERIFICATION PASSED:', m['verification']['passed'])

In [ ]:
# adversarial robustness with the fine-tuned model
!python predict.py --input adversarial/adversarial_tickets.csv \
    --artifacts artifacts_hf --outdir adv_hf
import pandas as pd
out = pd.read_csv('adv_hf/predictions.csv')
exp = pd.read_csv('adversarial/adversarial_tickets.csv')['Expected_Judgment']
print('ADVERSARIAL SCORE:', int((out['judgment']==exp).sum()), '/10')

In [ ]:
# zip + download the fine-tuned model and metrics
!zip -qr sia_hf_artifacts.zip artifacts_hf
from google.colab import files
files.download('sia_hf_artifacts.zip')

Upload `artifacts_hf/` to the GitHub repo (or a release) and the Streamlit app can serve the transformer backend by setting `SIA_ARTIFACTS=artifacts_hf`.